# GateIO — LOFO extras

Three additions that strengthen the leave-one-flight-out (LOFO) result:

1. **GateIO-LSTM on LOFO** — trains the recurrent baseline leakage-free on all 5 folds, so the learned baseline is evaluated the same honest way as GateIO (not only within-flight).
2. **Combined LOFO table + median/IQR** — one leakage-free table with GateIO, LSTM, EKF, Const-v side by side, plus median and IQR (5 folds have low statistical power, so spread matters).
3. **Trajectory figure** — a real predicted-vs-truth path for one held-out outage.

**Prerequisites:** the five fold files (`fold0.npz … fold4.npz`) already built with `data/preprocess/combine_baglevel.py`, and GateIO v2 already trained per fold (`checkpoints_lofo/fold{k}/gateio_v2_best.pt`). Edit the two paths in the setup cell to point at wherever your folds and checkpoints live on Drive.


## Setup (run once)

In [ ]:
import os, sys

REPO = "/content/gateio"                                       # cloned repo
FOLDS_DIR = "/content/drive/MyDrive/gateio/folds"              # fold0.npz .. fold4.npz
CKPT_DIR  = "/content/drive/MyDrive/gateio/checkpoints_lofo"   # foldK/gateio_v2_best.pt

# Mount Drive if the folds/checkpoints live there
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

if not os.path.isdir(REPO):
    !git clone https://github.com/nandini1612/gateio.git {REPO}
%cd {REPO}
!git pull            # refresh scripts if the repo was already cloned in this runtime
!pip -q install -r requirements.txt

assert all(os.path.exists(f"{FOLDS_DIR}/fold{k}.npz") for k in range(5)), \
    f"Missing fold npz files in {FOLDS_DIR}"
print("Folds and repo ready.")

## 1 — Train GateIO-LSTM on every LOFO fold

Same `--v2` residual head and settings as GateIO, only `--model lstm`. Writes `checkpoints_lofo/fold{k}/lstm_v2_best.pt` next to the GateIO checkpoints. Skips a fold if its LSTM checkpoint already exists, so it is safe to re-run.

In [ ]:
for k in range(5):
    ckpt = f"{CKPT_DIR}/fold{k}/lstm_v2_best.pt"
    if os.path.exists(ckpt):
        print(f"fold {k}: lstm_v2_best.pt already exists - skipping")
        continue
    print(f"
===== training GateIO-LSTM, fold {k} =====")
    !python -u train/train_gateio.py         --data {FOLDS_DIR}/fold{k}.npz         --model lstm --v2         --ckpt-dir {CKPT_DIR}/fold{k}

## 2 — Combined LOFO table (GateIO vs LSTM vs EKF vs Const-v) + median/IQR

`--lstm` adds the LSTM column; the evaluator also prints GateIO median and IQR for the pooled result. `--out` saves per-sequence rows for the paper's tables.

The bottom `LOFO POOLED` block is the headline table. Copy the `all` row into the preprint, and the `GateIO spread` line for the median + IQR.

In [ ]:
!python -u eval/evaluate_lofo.py     --folds-dir {FOLDS_DIR}     --ckpt-dir {CKPT_DIR}     --lstm     --out results/lofo_summary.csv

## 3 — Held-out trajectory figure

`--dual` draws two panels for the primary model (GateIO-LSTM): a typical straight outage and a turn, each at its group's median drift, so the figure shows both the common good case and the hard case rather than one large-error turn. Fold 2 (test = gnss03) has a good mix of straights and turns. Drop `--dual` and add `--prefer straight` for a single straight example, or `--model gateio` to plot the TCN backbone.

In [ ]:
!python -u eval/plot_lofo_trajectory.py \
    --folds-dir {FOLDS_DIR} \
    --ckpt-dir {CKPT_DIR} \
    --model lstm --fold 2 --dual \
    --out results/figures/fig_trajectory.png

from IPython.display import Image
Image("results/figures/fig_trajectory.png")

## After running

- `results/lofo_summary.csv` — per-sequence GateIO/LSTM/EKF/Const-v drifts (regenerated).
- `results/figures/fig_trajectory.png` — the new figure to embed in the README/preprint.

Then commit yourself:

```bash
git add results/lofo_summary.csv results/figures/fig_trajectory.png eval/evaluate_lofo.py eval/plot_lofo_trajectory.py notebooks/05_lofo_extras.ipynb
git commit -m "Add LOFO LSTM baseline, median/IQR reporting, and held-out trajectory figure"
git push
```
